# Phase 6: Investor Risk Profiling & Portfolio Diagnostics
**Financial Market Analytics & Investment Decision Support System for Indian Equities**
*CRISP-DM Phase: Modeling - Risk Capacity, Tolerance & Portfolio Analytics (PRD §11 & §12)*

---

### Educational & Analytical Context
This notebook implements and validates:
1. **Two-Tier Risk Profile Engine (PRD §11):** Capacity (Q1-Q6, horizon weighted 2x) and Tolerance (Q7-Q10) normalized to 0-100 and mapped to 5 risk tiers (Conservative to Aggressive), applying the **lower-of-two rule** and essential financial guardrails (emergency fund & investment horizon).
2. **Comprehensive Portfolio Diagnostics (PRD §12):** Asset allocation drift analysis against target models, single-stock and sector concentration metrics (Herfindahl-Hirschman Index), portfolio volatility, and the **Diversification Ratio**.
3. **Multi-Asset Fixture Walkthrough:** Rigorous evaluation across Bank-Heavy, Balanced Multi-Asset, and Fund-Only fixtures.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.profile.scoring import calculate_profile, load_questions
from src.portfolio.analytics import analyze_portfolio
from src.portfolio.holdings import parse_manual_holdings
from src.config import load_config

cfg = load_config()
print("System configuration and profiling modules loaded successfully.")

# 1. Load validated 14 questions
questions = load_questions()
print(f"Loaded {len(questions)} validated questions from questions.yaml")

# 2. Evaluate key personas
personas = {
    "Conservative (All Minimums)": {f"q{i}": 1 for i in range(1, 15)},
    "Aggressive (All Maximums)": {f"q{i}": 4 for i in range(1, 15)},
    "Lower-of-Two Rule (High Tolerance, Low Capacity)": {
        "q1": 1, "q2": 1, "q3": 1, "q4": 2, "q5": 1, "q6": 1, # Low capacity
        "q7": 4, "q8": 4, "q9": 4, "q10": 4,                  # High tolerance
        "q11": 2, "q12": "no", "q13": "no", "q14": "no"
    },
    "Emergency Fund Guardrail Active (< 3 months)": {
        "q1": 4, "q2": 4, "q3": 4, "q4": 1, "q5": 4, "q6": 4, # Emergency fund = 1 (< 3 mo)
        "q7": 4, "q8": 4, "q9": 4, "q10": 4,
        "q11": 2, "q12": "no", "q13": "no", "q14": "no"
    },
    "Horizon Guardrail Active (< 1 year)": {
        "q1": 4, "q2": 4, "q3": 4, "q4": 4, "q5": 4, "q6": 1, # Horizon = 1 (< 1 yr)
        "q7": 4, "q8": 4, "q9": 4, "q10": 4,
        "q11": 2, "q12": "no", "q13": "no", "q14": "no"
    }
}

profile_rows = []
for name, ans in personas.items():
    res = calculate_profile(ans, config=cfg, questions=questions)
    profile_rows.append({
        "Persona": name,
        "Capacity Score": f"{res.capacity_score:.1f} (L{res.capacity_level})",
        "Tolerance Score": f"{res.tolerance_score:.1f} (L{res.tolerance_level})",
        "Final Level": f"Level {res.final_level} ({res.label})",
        "Limiting Factor": res.limiting_factor,
        "Guardrail Triggered": len(res.guardrail_messages) > 0,
        "Target Equity %": f"{res.target_allocation.equity * 100:.0f}%"
    })

df_prof = pd.DataFrame(profile_rows)
display(df_prof)


System configuration and profiling modules loaded successfully.
Loaded 14 validated questions from questions.yaml


,Persona,Capacity Score,Tolerance Score,Final Level,Limiting Factor,Guardrail Triggered,Target Equity %
0,Conservative (All Minimums),0.0 (L1),0.0 (L1),Level 1 (Conservative),both,False,20%
1,Aggressive (All Maximums),100.0 (L5),100.0 (L5),Level 5 (Aggressive),both,False,85%
2,"Lower-of-Two Rule (High Tolerance, Low Capacity)",4.8 (L1),100.0 (L5),Level 1 (Conservative),capacity,False,20%
3,Emergency Fund Guardrail Active (< 3 months),85.7 (L5),100.0 (L5),Level 3 (Moderate),both,True,50%
4,Horizon Guardrail Active (< 1 year),71.4 (L4),100.0 (L5),Level 2 (Moderately Conservative),capacity,True,35%


In [2]:
# 3. Portfolio Diagnostics on Three Benchmark Fixtures
fixtures = {
    "Bank-Heavy Portfolio": {
        "stocks": [
            {"ticker": "HDFCBANK.NS", "quantity": 50, "avg_buy_price": 1500.0},
            {"ticker": "ICICIBANK.NS", "quantity": 40, "avg_buy_price": 1000.0},
            {"ticker": "KOTAKBANK.NS", "quantity": 25, "avg_buy_price": 1750.0},
            {"ticker": "AXISBANK.NS", "quantity": 30, "avg_buy_price": 1100.0},
            {"ticker": "SBIN.NS", "quantity": 50, "avg_buy_price": 750.0},
        ],
        "funds": []
    },
    "Balanced Multi-Asset Portfolio": {
        "stocks": [
            {"ticker": "RELIANCE.NS", "quantity": 20, "avg_buy_price": 1200.0},
            {"ticker": "TCS.NS", "quantity": 10, "avg_buy_price": 2400.0},
            {"ticker": "SUNPHARMA.NS", "quantity": 20, "avg_buy_price": 1150.0},
            {"ticker": "FEDERALBNK.NS", "quantity": 100, "avg_buy_price": 180.0},
            {"ticker": "VOLTAS.NS", "quantity": 15, "avg_buy_price": 950.0},
            {"ticker": "JINDALSAW.NS", "quantity": 25, "avg_buy_price": 500.0},
        ],
        "funds": [
            {"category": "debt", "amount": 80000.0},
            {"category": "gold", "amount": 20000.0}
        ]
    },
    "Fund-Only Portfolio": {
        "stocks": [],
        "funds": [
            {"category": "equity", "amount": 120000.0},
            {"category": "debt", "amount": 60000.0},
            {"category": "gold", "amount": 20000.0}
        ]
    }
}

target_l3 = cfg.target_allocations[3]

diag_rows = []
for name, data in fixtures.items():
    port = parse_manual_holdings(data["stocks"], data["funds"])
    an = analyze_portfolio(port, target_allocation=target_l3)
    
    top_sec = max(an.sector_weights.items(), key=lambda x: x[1]) if an.sector_weights else ("N/A", 0)
    drift_eq = an.drift.get("equity", 0.0) if an.drift else 0.0
    
    diag_rows.append({
        "Portfolio Fixture": name,
        "Total Valuation": f"₹{an.total_value:,.2f}",
        "Stock Count": len(an.stock_weights),
        "Annual Volatility": f"{an.volatility * 100:.2f}%" if an.volatility else "N/A",
        "Diversification Ratio": f"{an.diversification_ratio:.2f}" if an.diversification_ratio else "N/A",
        "Dominant Sector": f"{top_sec[0]} ({top_sec[1] * 100:.1f}%)",
        "Concentration Warnings": len(an.concentration_warnings),
        "Equity Drift vs L3 Target": f"{drift_eq * 100:+.1f}%"
    })

df_diag = pd.DataFrame(diag_rows)
display(df_diag)


,Portfolio Fixture,Total Valuation,Stock Count,Annual Volatility,Diversification Ratio,Dominant Sector,Concentration Warnings,Equity Drift vs L3 Target
0,Bank-Heavy Portfolio,"₹187,662.50",5,23.79%,1.23,Banking and Financials (100.0%),3,+50.0%
1,Balanced Multi-Asset Portfolio,"₹241,078.00",6,18.95%,1.59,Banking and Financials (24.7%),0,+8.5%
2,Fund-Only Portfolio,"₹200,000.00",0,N/A,1.00,N/A (0.0%),0,-50.0%


## 2. Key Methodological Takeaways
1. **Lower-of-Two Rule & Guardrails:** When an investor demonstrates high risk tolerance but limited financial capacity, the lower-of-two rule prevents dangerous overexposure. Emergency fund (< 3 months) and horizon (< 1 year) guardrails enforce strict caps at Level 3 and Level 2 respectively.
2. **Concentration Warning System:** The Bank-Heavy portfolio triggers concentration alerts with >90% Banking weight, while the Diversification Ratio confirms superior risk dispersion in the multi-asset portfolio.
